# Day 2 실습 — Advanced·Modular RAG + RAGAS 평가

## 들어가며

Day 1에서는 가장 기본형인 **Naive RAG** 파이프라인을 구현했습니다. 이번 실습에서는 한국어 QA 벤치마크 **KorQuAD v1** 위에서 **Advanced·Modular RAG** 의 핵심 기법(Multi-Query, RAG-Fusion, HyDE, Reranking, Self-RAG)을 단계적으로 적용하고, 그 결과를 **RAGAS** 로 정량 평가합니다.

이 실습이 끝나면 다음을 직접 말할 수 있습니다.

- Naive RAG 대비 **어떤 단계**를 보강하면 정답률이 올라가는가
- Multi-Query / RAG-Fusion / HyDE / Reranker / Self-RAG 는 각각 **어떤 코드 라인**으로 적용하는가
- RAGAS 의 4대 지표(Faithfulness · Answer Relevance · Context Precision · Context Recall)는 어떻게 계산되고 어떻게 읽는가
- 내 RAG 가 '얼마나 좋아졌는지'를 **숫자로** 보여주는 방법

### 목차

| 단계 | 내용 |
|---|---|
| Step 0 | 설치와 준비 |
| Step 1 | KorQuAD v1 Naive RAG 베이스라인 |
| Step 2 | Multi-Query Retrieval |
| Step 2.5 | RAG-Fusion (RRF 직접 구현) |
| Step 3 | HyDE |
| Step 4 | Cross-Encoder Reranking (multilingual) |
| Step 5 | Advanced RAG 체인 조립 |
| Step 5.5 | Self-RAG (검색 판단 + 자가 비평 + HyDE 재시도) |
| Step 6~7 | RAGAS 평가 데이터셋 + 4대 지표 비교 |
| Step 8 | (선택) 평가 데이터 자동 생성 |
| 추가 실습 A~K | KLUE-MRC (뉴스 도메인) 로 파이프라인 재구성 |

## Step 0 : 설치와 준비

Colab 기본 설치된 langchain 0.3 계열을 `ragas 0.2.10` 과 호환되는 `0.2.x` 조합으로 정리합니다. 처음 실행 시 3~5분 정도 걸립니다.

> ⚠️ **Colab 에서 아래 설치 셀을 실행한 뒤에는 [런타임 > 세션 다시 시작] 을 한 번 눌러주세요.** 재시작 후에는 설치 셀을 건너뛰고 그 아래 키 설정 셀부터 순서대로 실행하면 됩니다. (로컬/서버에서 이미 같은 버전이 설치돼 있으면 이 셀은 자동으로 건너뜁니다.)

In [1]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # 1) 기존 langchain / ragas 제거 — 버전 충돌로 인한 pip resolver 백트래킹 방지
    !pip uninstall -y ragas ragas-experimental langchain langchain-core langchain-community langchain-openai langchain-text-splitters langchain-chroma

    # 2) 0.2 시리즈 패치 버전까지 핀 설치 — resolver 부담 최소화
    !pip install --no-cache-dir \
        "ragas==0.2.10" \
        "langchain==0.2.17" \
        "langchain-core==0.2.43" \
        "langchain-community==0.2.19" \
        "langchain-openai==0.1.25" \
        "langchain-text-splitters==0.2.4" \
        "langchain-chroma==0.1.4" \
        pypdf chromadb tiktoken sentence-transformers datasets nest_asyncio pandas scipy
else:
    print("Colab 이 아니므로 설치 셀을 건너뜁니다. (동일한 핀 버전이 이미 설치돼 있어야 합니다)")

Colab 이 아니므로 설치 셀을 건너뜁니다. (동일한 핀 버전이 이미 설치돼 있어야 합니다)


### 키 설정 · 런타임 준비

OpenAI 키는 **코드에 직접 적지 않습니다.** Colab 에서는 좌측 🔑 [보안 비밀] 에 `OPENAI_KEY` 로 저장한 값을 읽어오고, 로컬/서버에서는 환경 변수 `OPENAI_API_KEY` 를 사용합니다.

In [2]:
import os

# chromadb 익명 통계 전송 끄기 — posthog SDK 인자 충돌로 ERROR 로그가 뜨는 것 방지
os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import nest_asyncio
nest_asyncio.apply()  # RAGAS 가 Colab 의 비동기 이벤트 루프와 충돌하지 않도록

if IN_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_KEY")

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 가 설정되지 않았습니다."
print("OPENAI_API_KEY 설정 완료 (길이: %d)" % len(os.environ["OPENAI_API_KEY"]))

OPENAI_API_KEY 설정 완료 (길이: 164)


## Step 1 : KorQuAD v1 위에서 Naive RAG 베이스라인 만들기

Day 1 에서 만든 RAG 파이프라인을 한국어 QA 벤치마크 **KorQuAD v1** 위에 다시 올립니다. 이후 단계는 모두 이 베이스라인에 '덧붙이는' 방식입니다.

- HuggingFace `datasets` 로 KorQuAD v1 자동 다운로드 (별도 PDF 업로드 불필요)
- 일부만 샘플링해 토큰 비용 통제
- Embedding → VectorStore → Retriever → LLM
- 검색 전략은 단순 `similarity` (top-k)

**📥 데이터셋**: <https://huggingface.co/datasets/KorQuAD/squad_kor_v1>

In [3]:
from datasets import load_dataset
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    """chunk 길이를 '문자 수'가 아니라 실제 LLM 토큰 수 기준으로 재기 위한 길이 함수."""
    return len(tokenizer.encode(text))

# 1) 데이터셋 로드 + 2000개 샘플링 + context 중복 제거 → unique 약 800개
#    (Vector DB 가 크면 Reranker 의 정밀도 개선 효과가 더 또렷하게 보입니다.)
raw_ds = load_dataset("squad_kor_v1", split="validation").shuffle(seed=42).select(range(2000))

unique = {}
for ex in raw_ds:
    if ex["context"] not in unique:
        unique[ex["context"]] = ex["title"]
context_docs = [Document(page_content=c, metadata={"title": t}) for c, t in unique.items()]

# 2) chunk 단위 분할 (KorQuAD context 는 짧지만 길이 균질화를 위해 splitter 사용)
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function=tiktoken_len)
docs = splitter.split_documents(context_docs)

# 3) Embedding & Chroma 적재 — 한 번에 넣으면 chromadb batch limit / OpenAI rate limit 에
#    걸릴 수 있어 100개씩 배치로 add_documents 합니다.
embedding = OpenAIEmbeddings(model="text-embedding-3-small")
db = Chroma(embedding_function=embedding, collection_name="korquad")
BATCH = 100
for i in range(0, len(docs), BATCH):
    db.add_documents(docs[i:i + BATCH])

# 4) Retriever (Naive: similarity)
naive_retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 3})

# 5) LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print(f"베이스라인 준비 완료 — unique context: {len(context_docs)}, chunks: {len(docs)}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


베이스라인 준비 완료 — unique context: 847, chunks: 1264


베이스라인 RAG 로 간단한 질의를 던져 답이 나오는지 확인합니다.

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

RAG_PROMPT = ChatPromptTemplate.from_template(
    "다음 문서를 참고해 질문에 한국어로 간결하게 답하세요. 문서에 없는 내용은 만들지 마세요.\n\n"
    "[문서]\n{context}\n\n"
    "[질문]\n{question}\n\n"
    "[답변]"
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

naive_chain = (
    {"context": naive_retriever | format_docs,
     "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

# 데이터셋에서 첫 질문 하나를 뽑아 테스트
TEST_Q = raw_ds[0]["question"]
print("Q:", TEST_Q)
print("A:", naive_chain.invoke(TEST_Q))
print("GT:", raw_ds[0]["answers"]["text"][0])

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Q: 2004년 이명박이 서울시장 재직시절 전면적으로 개선한 것은?


A: 대중교통체계입니다.
GT: 대중교통체계


## Step 2 : Pre-retrieval 강화 — Multi-Query Retrieval

사용자가 던진 질문 하나로만 검색하면 '다른 표현' 으로 적힌 정답을 놓칠 수 있습니다. **Multi-Query Retrieval** 은 LLM 에게 '같은 의도의 다른 질문 N개' 를 만들게 시킨 뒤, 각 질문으로 병렬 검색하고 결과를 합칩니다.

In [5]:
from langchain.retrievers.multi_query import MultiQueryRetriever
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=db.as_retriever(search_kwargs={"k": 3}),
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
)

# 어떤 '유사 질문' 으로 확장되는지 로그로 확인 가능
docs_mq = multi_query_retriever.invoke(TEST_Q)
print(f"검색된 문서 수: {len(docs_mq)}")
print("---")
print(docs_mq[0].page_content[:300])

INFO:langchain.retrievers.multi_query:Generated queries: ['2004년 이명박 서울시장이 재직할 때 어떤 주요 개선 사항이 있었나요?  ', '이명박이 2004년 서울시장으로 재직할 당시 추진한 주요 정책이나 변화는 무엇인가요?  ', '2004년 이명박 서울시장 시절에 이루어진 주요 개선 프로젝트는 어떤 것들이 있나요?']


검색된 문서 수: 6
---
2010년 한나라당 당내경선에서 나경원, 김충환 등의 경쟁자를 물리치고, 민선 5기 지방선거에서 서울시장 재선에 도전했다. 6월 2일에 치뤄진 지방선거에서 개표 초반에 한명숙 후보에게 뒤지다가, 후반 강남 3구의 개표가 시작되면서 역전하여 민선 5기 제34대 서울특별시장으로 재선되었다. 구체적으로 강남구(+59,206, +25.68%), 서초구(+43,820, +23.66%), 송파구(+23,814, +8.19%), 강동구(+11,097, +5.33%), 용산구(+8,579, +8.24%), 양천구(+1,078, +0.51%), 영


단일 질문(Naive, k=3)과 Multi-Query 의 검색 결과를 비교해 **검색 폭이 얼마나 넓어졌는지** 확인합니다.

In [6]:
docs_naive = naive_retriever.invoke(TEST_Q)
set_naive = {d.page_content for d in docs_naive}
set_mq = {d.page_content for d in docs_mq}

print(f"Naive top-3 문서 수      : {len(set_naive)}")
print(f"Multi-Query 합집합 문서 수 : {len(set_mq)}")
print(f"Multi-Query 만 찾아낸 문서 : {len(set_mq - set_naive)}개")

Naive top-3 문서 수      : 3
Multi-Query 합집합 문서 수 : 6
Multi-Query 만 찾아낸 문서 : 3개


## Step 2.5 : RAG-Fusion — Multi-Query + RRF 로 묶어내기

Step 2 의 Multi-Query 는 '유사 질문 N개로 병렬 검색' 까지만 했는데, **RAG-Fusion** 은 그 N개 검색 결과를 **Reciprocal Rank Fusion (RRF)** 공식으로 합쳐 '여러 쿼리에서 공통으로 상위에 떴던 문서' 를 최상단으로 끌어올립니다.

$$\text{score}(d) = \sum_{i=1}^{N} \frac{1}{k + \text{rank}_i(d)}$$

- $\text{rank}_i(d)$ : i번째 쿼리 결과에서 문서 $d$ 의 순위 (1부터)
- $k$ : 스무딩 상수 (관례적으로 60) — 상위 순위끼리의 점수 차를 완만하게 만들어 특정 쿼리의 1등이 결과를 독식하지 않게 합니다.

아래 셀에서 (1) sub-query 생성, (2) 각 sub-query 로 검색, (3) **RRF 구현**, (4) 결과 확인을 한 번에 진행합니다.

In [7]:
from collections import defaultdict

# (1) sub-query 생성 — Multi-Query 가 내부적으로 하는 일을 명시적으로 노출 (한국어)
SUBQUERY_PROMPT = ChatPromptTemplate.from_template(
    "당신은 검색 보조 AI 입니다. 다음 질문과 의미는 같지만 표현이 다른 4개의 한국어 검색 쿼리를 만드세요. "
    "오직 4개의 쿼리만 한 줄에 하나씩 출력하고, 번호나 다른 설명은 붙이지 마세요.\n\n질문: {question}"
)

def fan_out_queries(question, n=4):
    raw = (SUBQUERY_PROMPT | llm | StrOutputParser()).invoke({"question": question})
    return [q.strip() for q in raw.split("\n") if q.strip()][:n]


# (2) RRF 구현
def reciprocal_rank_fusion(results_per_query, k=60, top_k=3):
    """여러 쿼리의 검색 결과를 Reciprocal Rank Fusion 으로 하나의 랭킹으로 합칩니다.

    results_per_query : List[List[Document]]  쿼리별 검색 결과(순위 순).
    k                 : RRF smoothing 상수 (관례적으로 60).
    top_k             : 최종 반환할 문서 개수.

    점수는 유사도 값이 아니라 '순위' 만 사용하므로, 쿼리마다 점수 스케일이 달라도
    안전하게 합칠 수 있는 것이 RRF 의 장점입니다.
    """
    scores = defaultdict(float)
    docs_by_key = {}

    for docs in results_per_query:
        for rank, doc in enumerate(docs):  # rank 는 0부터
            key = doc.page_content
            scores[key] += 1.0 / (k + rank + 1)
            docs_by_key[key] = doc

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [docs_by_key[key] for key, _ in ranked[:top_k]]


# (3) 한 번 돌려보기
sub_queries = fan_out_queries(TEST_Q)
print(f"확장 질문 {len(sub_queries)}개:")
for q in sub_queries:
    print(" -", q)

results_per_q = [db.similarity_search(q, k=5) for q in sub_queries]
fused = reciprocal_rank_fusion(results_per_q, k=60, top_k=3)

print("\nRAG-Fusion top-1 문서:")
print(fused[0].page_content[:300] if fused else "(결과 없음)")

확장 질문 4개:
 - 2004년 이명박 서울시장 재직 중 개선한 사항은?
 - 이명박이 2004년 서울시장으로서 개선한 내용은 무엇인가?
 - 2004년 서울시장 이명박이 전면적으로 개선한 것은 어떤 것인가?
 - 이명박 서울시장 재직 시절 2004년에 개선한 것은 무엇인지?



RAG-Fusion top-1 문서:
2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도 했다. 하지만 새 교통체계가 정착되면서 많은 긍정적인 효과를 가져오게 된다. 중앙버스차로 도입으로 버스의 평균 속도가 증가하여 정시에 도착하는 빈도가 늘어났고 환승제도로 인한 교


RRF 가 실제로 어떤 점수를 매겼는지, 그리고 '여러 쿼리에서 공통으로 등장한 문서' 가 위로 올라오는지 확인합니다.

In [8]:
scores_dbg = defaultdict(float)
appear = defaultdict(int)
for docs_i in results_per_q:
    for rank, d in enumerate(docs_i):
        scores_dbg[d.page_content] += 1.0 / (60 + rank + 1)
        appear[d.page_content] += 1

for i, (key, s) in enumerate(sorted(scores_dbg.items(), key=lambda x: x[1], reverse=True)[:5], 1):
    print(f"{i}. RRF={s:.5f}  등장 쿼리 수={appear[key]}/{len(results_per_q)}  {key[:60].strip()}...")

1. RRF=0.06557  등장 쿼리 수=4/4  2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이...
2. RRF=0.06452  등장 쿼리 수=4/4  2010년 한나라당 당내경선에서 나경원, 김충환 등의 경쟁자를 물리치고, 민선 5기 지방선거에서 서울시장 재...
3. RRF=0.04762  등장 쿼리 수=3/4  사법연수원을 16기로 수료한 후 변호사 생활을 하다가 16대 총선에서 서울 강남을에 출마하여 정계에 입문했다...
4. RRF=0.03126  등장 쿼리 수=2/4  2009년 1월 초 박대성이 체포되자, 일부 언론들은 보도에서 박대성의 실명, 얼굴, 주민등록번호 등을 노출...
5. RRF=0.03125  등장 쿼리 수=2/4  1995년 초 그는 내무부 장관에게 특별 지시를 내려 1991년 이후 실시되던 지방 자치 제도를 확대시켜,...


## Step 3 : 패턴 ② HyDE — 가상의 '정답' 으로 진짜 정답 찾기

질문은 짧은 의문문, 정답은 긴 평서문이라 둘의 임베딩이 의외로 멀 수 있습니다. **HyDE(Hypothetical Document Embeddings)** 는 검색 전에 LLM 에게 '가상의 정답' 을 쓰게 한 뒤, 그 가상 답변을 임베딩해서 검색합니다.

In [9]:
HYDE_PROMPT = ChatPromptTemplate.from_template(
    "당신은 해당 분야 전문가입니다. 다음 질문에 대해 그럴듯한 한국어 답변 한 문단을 작성하세요. "
    "확실하지 않다면 가장 합리적인 추측을 적어주세요.\n\n"
    "질문: {question}\n\n가상 답변:"
)

hyde_generator = HYDE_PROMPT | llm | StrOutputParser()

def hyde_retrieve(question, k=3):
    """질문 → 가상의 답변 → 가상 답변을 임베딩해 검색"""
    hypothetical = hyde_generator.invoke({"question": question})
    return db.similarity_search(hypothetical, k=k), hypothetical

docs_hyde, hyp = hyde_retrieve(TEST_Q)
print("가상 답변(HyDE):\n", hyp[:300], "\n---")
print("검색된 문서 수:", len(docs_hyde))
print("첫 문서:", docs_hyde[0].page_content[:200])

가상 답변(HyDE):
 2004년 이명박이 서울시장으로 재직하던 시절, 그는 서울시의 교통 체계를 전면적으로 개선하는 데 주력했습니다. 특히, 그는 '서울시 교통체계 개선 종합계획'을 수립하여 대중교통의 효율성을 높이고, 도로 혼잡을 줄이기 위한 다양한 정책을 시행했습니다. 이 과정에서 지하철 노선 확장과 버스 전용차선 도입, 그리고 자전거 도로의 확충 등이 이루어졌습니다. 이러한 노력은 서울시민의 교통 편의성을 높이고, 대기오염 문제를 완화하는 데 기여했습니다. 
---
검색된 문서 수: 3
첫 문서: 2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도


## Step 4 : Post-retrieval 강화 — Cross-Encoder Reranking (multilingual)

검색 결과를 그대로 LLM 에 넘기지 않고, **Cross-encoder reranker** 가 (질문, 문단) 을 함께 보면서 진짜 관련도를 다시 점수화합니다. 정밀도가 15~30% 개선되는 게 일반적인 보고입니다.

한국어 문서를 다루므로 다국어 cross-encoder `BAAI/bge-reranker-v2-m3` 를 사용합니다. 처음 실행 시 모델 다운로드(~2GB)가 발생합니다.

In [10]:
from sentence_transformers import CrossEncoder

# 다국어 cross-encoder (한국어 포함). CPU 에서도 동작하지만 GPU 가 있으면 훨씬 빠릅니다.
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512)

def rerank(query, docs, top_k=3):
    """검색된 docs 를 cross-encoder 로 다시 점수화해 상위 top_k 만 반환.

    bi-encoder(임베딩 검색)는 질문과 문서를 따로 벡터화하지만, cross-encoder 는
    (질문, 문서) 쌍을 한 번에 인코딩하므로 관련도 판단이 훨씬 정밀합니다.
    대신 후보 수만큼 forward pass 가 필요해 느리므로 '넓게 검색 → 좁게 재정렬' 로 씁니다.
    """
    if not docs:
        return []
    pairs = [(query, d.page_content) for d in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [d for d, _ in ranked[:top_k]]

candidates = db.as_retriever(search_kwargs={"k": 10}).invoke(TEST_Q)
top3 = rerank(TEST_Q, candidates, top_k=3)
print(f"후보 {len(candidates)}개 → Reranker 로 상위 3개 선별")
print("최상위 문서:", top3[0].page_content[:200])

후보 10개 → Reranker 로 상위 3개 선별
최상위 문서: 2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도


Reranker 가 임베딩 검색 순위를 실제로 얼마나 바꾸는지 확인합니다. (원래 몇 등이던 문서가 1등이 되었는지)

In [11]:
pairs = [(TEST_Q, d.page_content) for d in candidates]
sc = reranker.predict(pairs)
order = sorted(range(len(candidates)), key=lambda i: sc[i], reverse=True)
print("rerank 후 순위 → (원래 임베딩 순위, 점수)")
for new_rank, i in enumerate(order[:5], 1):
    print(f"  {new_rank}위 ← 원래 {i+1}위 (score={sc[i]:.3f})  {candidates[i].page_content[:50].strip()}...")

rerank 후 순위 → (원래 임베딩 순위, 점수)
  1위 ← 원래 1위 (score=1.000)  2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교...
  2위 ← 원래 3위 (score=0.500)  2004년 이명박 전 서울시장의 대중교통 정책으로 서울시민은 대중교통 환승시 무료나 할인된...
  3위 ← 원래 5위 (score=0.010)  사법연수원을 16기로 수료한 후 변호사 생활을 하다가 16대 총선에서 서울 강남을에 출마하...
  4위 ← 원래 4위 (score=0.004)  1995년 초 그는 내무부 장관에게 특별 지시를 내려 1991년 이후 실시되던 지방 자치...
  5위 ← 원래 6위 (score=0.002)  또한 2000년대 들어 시장경제의 지속적인 발전에 따라 민법전 제정의 목표를 정하고 단계적...


## Step 5 : Advanced RAG 체인 조립

위에서 만든 컴포넌트를 하나의 체인으로 묶습니다. **'넓게 검색 → Reranker 로 좁히기 → LLM 답변'** 패턴이 가장 흔히 쓰입니다.

In [12]:
def advanced_rag(question):
    # 1) 후보를 넓게 검색 (k=10)
    candidates = db.as_retriever(search_kwargs={"k": 10}).invoke(question)
    # 2) Cross-encoder 로 진짜 관련도 재정렬 후 상위 3개
    top = rerank(question, candidates, top_k=3)
    # 3) 프롬프트에 컨텍스트로 주입 → 답변
    context = format_docs(top)
    answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": context, "question": question})
    return answer, top

ans_adv, ctx_adv = advanced_rag(TEST_Q)
print("Advanced RAG 답변:\n", ans_adv)

Advanced RAG 답변:
 대중교통체계입니다.


### (참고) RAG-Fusion 체인도 함께 만들어 두기

Step 2.5 의 RRF 를 그대로 검색기로 쓰는 변형입니다. 뒤의 RAGAS 평가에서 세 번째 파이프라인으로 비교합니다.

In [13]:
def fusion_rag(question, top_k=3):
    """Multi-Query 로 fan-out → 쿼리별 검색 → RRF 로 융합 → LLM 답변"""
    queries = [question] + fan_out_queries(question)
    results = [db.similarity_search(q, k=5) for q in queries]
    top = reciprocal_rank_fusion(results, k=60, top_k=top_k)
    answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": format_docs(top), "question": question})
    return answer, top

ans_fus, ctx_fus = fusion_rag(TEST_Q)
print("RAG-Fusion 답변:\n", ans_fus)

RAG-Fusion 답변:
 대중교통체계입니다.


## Step 5.5 : Self-RAG — 검색 필요성 판단 + 답변 자가 비평

Self-RAG 의 핵심은 **LLM 이 검색·답변 과정에 스스로 비평(critique)을 끼워 넣는다** 는 점입니다. 공식 Self-RAG 모델을 받지 않고 **세 개의 작은 LLM 프롬프트** 로 같은 흐름을 흉내냅니다.

1. **Retrieve 결정** — 외부 검색이 필요한지 판단 (`YES`/`NO`)
2. **답변 생성** — `YES` 면 일반 RAG, `NO` 면 검색 없이 LLM 단독 답변
3. **답변 자가 비평** — 답변이 컨텍스트에 근거하는지 점검 (`SUPPORTED`/`NOT_SUPPORTED`)
4. **보완 재시도** — `NOT_SUPPORTED` 면 HyDE 로 검색 쿼리를 바꿔 한 번 더 시도

In [14]:
# Self-RAG : retrieve 판단 + 자가 비평 + HyDE 재시도

# (1) 검색 필요성 판단 프롬프트
RETRIEVE_DECISION_PROMPT = ChatPromptTemplate.from_template(
    "당신은 RAG 시스템의 검색 라우터입니다. 아래 질문에 답하기 위해 외부 문서 검색이 필요한지 판단하세요.\n"
    "- 특정 인물·사건·연도·고유명사 등 문서에서 사실을 확인해야 하면: YES\n"
    "- 일반 상식, 단순 계산, 용어 정의처럼 모델이 가진 지식만으로 답할 수 있으면: NO\n"
    "설명 없이 YES 또는 NO 한 단어만 출력하세요.\n\n"
    "질문: {question}\n\n판단:"
)

# (2) 답변 자가 비평 프롬프트
CRITIQUE_PROMPT = ChatPromptTemplate.from_template(
    "당신은 사실 검증자입니다. [답변] 의 내용이 [문서] 안에서 확인되는지 판단하세요.\n"
    "- 답변이 한 단어·한 구절처럼 짧아도, 그 내용이 문서에 나타나 있으면: SUPPORTED\n"
    "- 답변에 문서에 없는 사실이 섞여 있거나 문서로 확인할 수 없으면: NOT_SUPPORTED\n"
    "질문에 대한 완결성이 아니라 '문서 근거 여부' 만 봅니다. 설명 없이 한 단어만 출력하세요.\n\n"
    "[문서]\n{context}\n\n[답변]\n{answer}\n\n판정:"
)


def self_rag(question, max_retries=1, verbose=True):
    decision = (RETRIEVE_DECISION_PROMPT | llm | StrOutputParser()).invoke(
        {"question": question}).strip().upper()
    if verbose:
        print(f"[1] Retrieve 필요? -> {decision}")

    if decision.startswith("NO"):
        ans = llm.invoke(question).content
        if verbose:
            print("[2] LLM 단독 답변 사용")
        return ans, []

    docs = db.as_retriever(search_kwargs={"k": 3}).invoke(question)

    for attempt in range(max_retries + 1):
        answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
            {"context": format_docs(docs), "question": question})
        critique = (CRITIQUE_PROMPT | llm | StrOutputParser()).invoke(
            {"context": format_docs(docs), "answer": answer}).strip().upper()
        if verbose:
            print(f"[3] 시도 {attempt+1} — 자가 비평: {critique}")

        if "NOT" not in critique:
            return answer, docs

        if attempt < max_retries:
            hyp = hyde_generator.invoke({"question": question})
            docs = db.similarity_search(hyp, k=3)
            if verbose:
                print("[4] NOT_SUPPORTED -> HyDE 가상 답변으로 재검색")

    return answer, docs


ans_sr, ctx_sr = self_rag(TEST_Q)
print("\n=== Self-RAG 최종 답변 ===")
print(ans_sr)

[1] Retrieve 필요? -> YES


[3] 시도 1 — 자가 비평: SUPPORTED

=== Self-RAG 최종 답변 ===
대중교통체계입니다.


라우터가 실제로 잘 동작하는지, **검색이 필요 없는 질문** 과 **검색이 필요한 질문** 을 각각 넣어 확인합니다.

In [15]:
print("### 검색이 필요 없는 질문")
a1, c1 = self_rag("2 더하기 3은 얼마인가요?")
print("답변:", a1.strip()[:120], "| 사용한 컨텍스트 수:", len(c1))

print("\n### 검색이 필요한 질문")
a2, c2 = self_rag(raw_ds[1]["question"])
print("질문:", raw_ds[1]["question"])
print("답변:", a2.strip()[:120], "| 사용한 컨텍스트 수:", len(c2))

### 검색이 필요 없는 질문


[1] Retrieve 필요? -> NO


[2] LLM 단독 답변 사용
답변: 2 더하기 3은 5입니다. | 사용한 컨텍스트 수: 0

### 검색이 필요한 질문


[1] Retrieve 필요? -> YES


[3] 시도 1 — 자가 비평: NOT_SUPPORTED


[4] NOT_SUPPORTED -> HyDE 가상 답변으로 재검색


[3] 시도 2 — 자가 비평: NOT_SUPPORTED
질문: 11월 24일 김영삼이 대통령 명령으로 제정한 법은?
답변: 11월 24일 김영삼이 대통령 명령으로 제정한 법은 '국가보안법'입니다. | 사용한 컨텍스트 수: 3


## Step 6 : RAGAS 평가용 데이터셋 만들기

RAGAS 는 네 가지 자료가 필요합니다.

- `user_input` — 사용자 질문
- `response` — RAG 가 생성한 답변
- `retrieved_contexts` — RAG 가 참고한 문서들
- `reference` — 모범 답안 (Ground Truth)

**KorQuAD 는 사람이 작성한 정답이 이미 포함**되어 있어 `reference` 를 그대로 가져다 씁니다. 같은 질문 셋을 **Naive / RAG-Fusion / Advanced** 세 파이프라인으로 풀어 비교합니다.

In [16]:
# 평가용 질문/정답 자동 추출 (KorQuAD)
EVAL_N = 20  # 평가 질문 수. 표본 분산을 줄이려 20개로 설정. 줄이려면 5~10.
eval_samples = list(raw_ds)[:EVAL_N]
questions = [ex["question"] for ex in eval_samples]
ground_truths = [ex["answers"]["text"][0] for ex in eval_samples]

# Naive RAG 로 답변 + 컨텍스트 수집
naive_answers, naive_contexts = [], []
for q in questions:
    ctx = naive_retriever.invoke(q)
    a = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": format_docs(ctx), "question": q})
    naive_answers.append(a)
    naive_contexts.append([d.page_content for d in ctx])

# Advanced RAG (넓게 검색 → Rerank) 로 답변 + 컨텍스트 수집
adv_answers, adv_contexts = [], []
for q in questions:
    a, ctx = advanced_rag(q)
    adv_answers.append(a)
    adv_contexts.append([d.page_content for d in ctx])

# RAG-Fusion (Multi-Query + RRF) 로 답변 + 컨텍스트 수집
fus_answers, fus_contexts = [], []
for q in questions:
    a, ctx = fusion_rag(q)
    fus_answers.append(a)
    fus_contexts.append([d.page_content for d in ctx])

print(f"데이터셋 준비 완료 — {EVAL_N}개 질문 × 3개 파이프라인")

데이터셋 준비 완료 — 20개 질문 × 3개 파이프라인


In [17]:
from datasets import Dataset

def make_dataset(answers, contexts, qs=None, refs=None):
    return Dataset.from_dict({
        "user_input":         qs if qs is not None else questions,
        "response":           answers,
        "retrieved_contexts": contexts,
        "reference":          refs if refs is not None else ground_truths,
    })

naive_ds = make_dataset(naive_answers, naive_contexts)
adv_ds   = make_dataset(adv_answers,   adv_contexts)
fus_ds   = make_dataset(fus_answers,   fus_contexts)
print(naive_ds)

Dataset({
    features: ['user_input', 'response', 'retrieved_contexts', 'reference'],
    num_rows: 20
})


## Step 7 : RAGAS 로 4대 지표 계산하기

Judge LLM 은 `gpt-4o-mini`, 임베딩은 `text-embedding-3-small` 로 설정합니다. (Judge 에 더 강한 모델을 쓰면 채점은 정교해지지만 비용이 늘어납니다.)

In [18]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness, answer_relevancy,
    context_precision, context_recall,
)

judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
judge_emb = OpenAIEmbeddings(model="text-embedding-3-small")
metrics = [faithfulness, answer_relevancy, context_precision, context_recall]

def run_eval(ds, label):
    print(f"=== {label} 채점 ===")
    return evaluate(ds, metrics=metrics, llm=judge_llm, embeddings=judge_emb,
                    raise_exceptions=False)

naive_result = run_eval(naive_ds, "Naive RAG")
fus_result   = run_eval(fus_ds,   "RAG-Fusion")
adv_result   = run_eval(adv_ds,   "Advanced RAG")

=== Naive RAG 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

=== RAG-Fusion 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

=== Advanced RAG 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

In [19]:
import pandas as pd

METRIC_COLS = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]

naive_df = naive_result.to_pandas()
fus_df   = fus_result.to_pandas()
adv_df   = adv_result.to_pandas()

def summary(df, label):
    avg = df[METRIC_COLS].mean()
    avg.name = label
    return avg

compare = pd.concat([summary(naive_df, "Naive RAG"),
                     summary(fus_df,   "RAG-Fusion"),
                     summary(adv_df,   "Advanced RAG")], axis=1)
print(compare.round(3))
print("\nDelta (Advanced - Naive):")
print((compare["Advanced RAG"] - compare["Naive RAG"]).round(3))
print("\nDelta (RAG-Fusion - Naive):")
print((compare["RAG-Fusion"] - compare["Naive RAG"]).round(3))

                   Naive RAG  RAG-Fusion  Advanced RAG
faithfulness           0.600       0.725         0.850
answer_relevancy       0.261       0.269         0.284
context_precision      0.642       0.692         0.800
context_recall         0.700       0.800         0.800

Delta (Advanced - Naive):
faithfulness         0.250
answer_relevancy     0.023
context_precision    0.158
context_recall       0.100
dtype: float64

Delta (RAG-Fusion - Naive):
faithfulness         0.125
answer_relevancy     0.009
context_precision    0.050
context_recall       0.100
dtype: float64


### 결과 해석 가이드

비교표를 처음 보면 **'Advanced 가 더 나쁜 거 아닌가?'** 라는 착각을 하기 쉽습니다. KorQuAD 위에서의 해석 방법입니다.

1. **`context_precision` 의 개선(+)이 Advanced RAG 의 핵심 효과** — Reranker 가 정답 문단을 상단에 올렸다는 의미. Δ가 0.05~0.15 면 잘 작동.
2. **`context_recall` 은 1.0 으로 포화될 수 있다** — unique context 가 수백 개 수준이면 Naive top-3 에도 정답이 거의 항상 들어옵니다. 수만 문서 규모에서 차이가 드러납니다.
3. **`faithfulness` 가 살짝 떨어질 수 있다** — 컨텍스트가 '짧고 집중' 되면 일부 주장이 미뒷받침으로 채점될 수 있음 (-0.1 이내면 정상).
4. **`answer_relevancy` 가 낮은 이유** — KorQuAD 정답이 한 단어~한 구절이라 RAGAS 가 답변에서 질문을 역추론할 때 흐려집니다. 데이터셋 특성이지 모델 잘못이 아닙니다.
5. **표본 20개에서 Δ가 ±0.05 이내면 '차이 없음'** 으로 봐야 합니다. 아래에서 paired t-test 로 확인합니다.

In [20]:
from scipy import stats

print("Naive vs Advanced — paired t-test (표본 20)")
for m in METRIC_COLS:
    a = naive_df[m].astype(float).fillna(0)
    b = adv_df[m].astype(float).fillna(0)
    diff = b.mean() - a.mean()
    if (b - a).abs().sum() == 0:
        print(f"  {m:<18} Δ={diff:+.3f}  (두 파이프라인 결과가 동일 — 검정 불가)")
        continue
    t, p = stats.ttest_rel(b, a)
    verdict = "유의미(p<0.05)" if p < 0.05 else "표본 noise 수준"
    print(f"  {m:<18} Δ={diff:+.3f}  t={t:+.2f}  p={p:.3f}  → {verdict}")

Naive vs Advanced — paired t-test (표본 20)
  faithfulness       Δ=+0.250  t=+2.52  p=0.021  → 유의미(p<0.05)
  answer_relevancy   Δ=+0.023  t=+0.65  p=0.525  → 표본 noise 수준
  context_precision  Δ=+0.158  t=+2.17  p=0.043  → 유의미(p<0.05)
  context_recall     Δ=+0.100  t=+1.45  p=0.163  → 표본 noise 수준


### Quiz

**Q. 위 표에서 Advanced RAG 가 가장 크게 개선한 지표는 무엇이고, 어떤 기법과 직접 연결될까요?**

**A.** 보통 `context_precision` 이 가장 크게 오릅니다. 이는 **Cross-encoder Reranker** 가 '진짜 관련도가 높은 문서를 상위에 두는 일' 을 했다는 뜻입니다. `context_recall` 은 **Multi-Query / RAG-Fusion** 이 검색 폭을 넓혔을 때 함께 오르고, `faithfulness` 와 `answer_relevancy` 는 컨텍스트 품질이 올라가면 부수적으로 개선됩니다.

## Step 8 : (선택) 평가 데이터를 LLM 으로 자동 생성하기

현업에서는 모범 답안(`reference`)을 사람이 작성하는 게 가장 큰 부담입니다. RAGAS 는 **원본 문서만 주면 Question·Reference·Context 세트를 자동 생성** 하는 기능을 제공합니다.

공식 문서: <https://docs.ragas.io/en/stable/getstarted/rag_testset_generation/>

```python
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

generator = TestsetGenerator(
    llm=LangchainLLMWrapper(judge_llm),
    embedding_model=LangchainEmbeddingsWrapper(judge_emb),
)
testset = generator.generate_with_langchain_docs(context_docs[:20], testset_size=5)
testset.to_pandas()
```

> 비용 때문에 이 노트북에서는 실행하지 않고 코드만 남겨둡니다. KorQuAD 처럼 사람이 만든 정답이 있는 경우에는 그대로 쓰는 편이 더 정확합니다.

---

# 추가 실습 — KLUE-MRC 한국어 뉴스 MRC 벤치마크로 RAG 평가하기

메인 실습은 위키 기반 **KorQuAD v1** 으로 진행했습니다. 이번에는 도메인을 바꿔 **한국어 뉴스 기사 기반 KLUE-MRC** 위에서 같은 파이프라인을 처음부터 다시 조립합니다.

- 한국어 **뉴스 기사** 기반 (KorQuAD 의 위키와 도메인이 다름)
- 사람이 작성한 정답 포함
- **`is_impossible=True`** 인 답할 수 없는 질문도 섞여 있어 필터링 필요

**📥 데이터셋**: <https://huggingface.co/datasets/klue> · <https://klue-benchmark.com/> · <https://arxiv.org/abs/2105.09680>

### Step A. 데이터셋 로드

In [21]:
from datasets import load_dataset

ds_klue = load_dataset("klue", "mrc", split="validation")
print(ds_klue)
print("\n--- 샘플 1건 ---")
sample0 = {k: ds_klue[0][k] for k in ds_klue.column_names}
for k, v in sample0.items():
    v = str(v)
    print(f"{k}: {v[:200]}{'...' if len(v) > 200 else ''}")

Dataset({
    features: ['title', 'context', 'news_category', 'source', 'guid', 'is_impossible', 'question_type', 'question', 'answers'],
    num_rows: 5841
})

--- 샘플 1건 ---
title: BMW 코리아, 창립 25주년 기념 ‘BMW 코리아 25주년 에디션’ 한정 출시
context: BMW 코리아(대표 한상윤)는 창립 25주년을 기념하는 ‘BMW 코리아 25주년 에디션’을 한정 출시한다고 밝혔다. 이번 BMW 코리아 25주년 에디션(이하 25주년 에디션)은 BMW 3시리즈와 5시리즈, 7시리즈, 8시리즈 총 4종, 6개 모델로 출시되며, BMW 클래식 모델들로 선보인 바 있는 헤리티지 컬러가 차체에 적용돼 레트로한 느낌과 신구의 조화가...
news_category: 자동차
source: acrofan
guid: klue-mrc-v1_dev_01891
is_impossible: False
question_type: 2
question: 말라카이트에서 나온 색깔을 사용한 에디션은?
answers: {'answer_start': [666, 666], 'text': ['뉴 740Li 25주년 에디션', '뉴 740Li 25주년']}


### Step B. Context 추출 + 중복 제거 (+ `is_impossible` 필터링)

KLUE-MRC 에는 **`is_impossible=True`** 케이스(= context 만 보고는 답할 수 없는 질문)가 섞여 있습니다. ground truth 가 비어 있으면 RAGAS 의 `context_recall` 이 깨지므로 답이 있는 샘플만 남깁니다.

In [22]:
from langchain_core.documents import Document

# 1) 답할 수 있는 샘플만 남기고 300개 샘플링
answerable = ds_klue.filter(lambda x: not x["is_impossible"]).shuffle(seed=42).select(range(300))

# 2) context 기준 중복 제거 → Document 로 감싸기
unique_klue = {}
for ex in answerable:
    if ex["context"] not in unique_klue:
        unique_klue[ex["context"]] = ex["title"]
context_docs_klue = [Document(page_content=c, metadata={"title": t})
                     for c, t in unique_klue.items()]

print("샘플 수:", len(answerable), "| unique context 수:", len(context_docs_klue))
print("평균 토큰 수:", int(sum(tiktoken_len(d.page_content) for d in context_docs_klue) / len(context_docs_klue)))
print("\n첫 문서 미리보기:\n", context_docs_klue[0].page_content[:200])

샘플 수: 300 | unique context 수: 299
평균 토큰 수: 1063

첫 문서 미리보기:
 국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 20


### Step C. Embedding + VectorStore

메인 실습의 `embedding` 을 재사용해 새 Chroma DB `db_klue` 를 만듭니다. (메인의 `db` 는 비교를 위해 덮어쓰지 않습니다.)

> ⚠️ 뉴스 context 는 평균 토큰 수가 커서 150개 이상을 한 번에 넘기면 OpenAI embeddings 의 **300k 토큰/요청 한도** 에 걸립니다. 100개씩 batch 로 `add_documents` 하세요.

In [23]:
db_klue = Chroma(embedding_function=embedding, collection_name="klue_mrc")
BATCH = 100
for i in range(0, len(context_docs_klue), BATCH):
    db_klue.add_documents(context_docs_klue[i:i + BATCH])

print("db_klue 적재 완료 —", db_klue._collection.count(), "chunks")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


db_klue 적재 완료 — 299 chunks


### Step D. 평가용 질문/정답 세트 추출

Step B 에서 필터링·샘플링한 데이터 중 앞에서 20개를 평가용으로 떼어냅니다.

In [24]:
EVAL_N_KLUE = 20
eval_klue = list(answerable)[:EVAL_N_KLUE]
questions_klue = [ex["question"] for ex in eval_klue]
ground_truths_klue = [ex["answers"]["text"][0] for ex in eval_klue]

print(len(questions_klue), len(ground_truths_klue))
for q, g in list(zip(questions_klue, ground_truths_klue))[:3]:
    print(f"- Q: {q}\n  GT: {g}")

20 20
- Q: 국내에서 해킹을 당한 리플이 들어간 통장의 갯수는?
  GT: 두 개
- Q: 정유공장 공사는 어느 도시에서 진행되는가?
  GT: 카르발라
- Q: 가장 먼저 리그 진출 팀이 결정되는 경기의 시작 시간은 언제인가?
  GT: 오후 5시 45분


### Step E. Naive RAG 베이스라인 (KLUE)

뉴스 도메인 특성을 살려 '기사 본문에 근거해서만 답하라' 는 지시를 추가한 프롬프트를 사용합니다.

In [25]:
RAG_PROMPT_KLUE = ChatPromptTemplate.from_template(
    "다음 뉴스 기사 본문만 참고해 질문에 한국어로 간결하게 답하세요. "
    "기사에 없는 내용은 추측하지 말고, 기사에서 찾을 수 없으면 '기사에서 확인할 수 없습니다' 라고 답하세요.\n\n"
    "[기사]\n{context}\n\n[질문]\n{question}\n\n[답변]"
)

naive_retriever_klue = db_klue.as_retriever(search_type="similarity", search_kwargs={"k": 3})

naive_chain_klue = (
    {"context": naive_retriever_klue | format_docs,
     "question": RunnablePassthrough()}
    | RAG_PROMPT_KLUE
    | llm
    | StrOutputParser()
)

TEST_Q_KLUE = questions_klue[0]
print("Q:", TEST_Q_KLUE)
print("A:", naive_chain_klue.invoke(TEST_Q_KLUE))
print("GT:", ground_truths_klue[0])

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Q: 국내에서 해킹을 당한 리플이 들어간 통장의 갯수는?


A: 200여 개의 계좌입니다.
GT: 두 개


### Step F. Multi-Query Retrieval (KLUE)

확장 질문 로깅을 켜서 어떤 한국어 변형 질문이 만들어지는지 확인합니다.

In [26]:
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

multi_query_retriever_klue = MultiQueryRetriever.from_llm(
    retriever=db_klue.as_retriever(search_kwargs={"k": 3}),
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
)

docs_mq_klue = multi_query_retriever_klue.invoke(TEST_Q_KLUE)
print("검색된 문서 수:", len(docs_mq_klue))
print("첫 문서:", docs_mq_klue[0].page_content[:200])

INFO:langchain.retrievers.multi_query:Generated queries: ['1. 국내에서 해킹으로 피해를 입은 리플이 포함된 통장의 수는 몇 개인가요?  ', '2. 한국에서 해킹 사건에 연루된 리플이 있는 통장 수는 얼마인가요?  ', '3. 국내에서 해킹으로 영향을 받은 리플이 포함된 계좌의 개수는 어떻게 되나요?']


검색된 문서 수: 6
첫 문서: 국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 20


### Step G. HyDE (KLUE)

뉴스 도메인에 맞춰 '기자가 쓴 기사 한 문단' 형태의 가상 답변을 생성합니다.

In [27]:
HYDE_PROMPT_KLUE = ChatPromptTemplate.from_template(
    "당신은 한국 언론사의 기자입니다. 다음 질문에 답하는 뉴스 기사 본문 한 문단을 작성하세요. "
    "기사체(평서문)로 쓰고, 확실하지 않으면 가장 그럴듯한 내용을 적으세요.\n\n"
    "질문: {question}\n\n기사 본문:"
)

hyde_generator_klue = HYDE_PROMPT_KLUE | llm | StrOutputParser()

def hyde_retrieve_klue(question, k=3):
    """질문 → 가상의 기사 본문 → 그 본문을 임베딩해 db_klue 에서 검색"""
    hypothetical = hyde_generator_klue.invoke({"question": question})
    return db_klue.similarity_search(hypothetical, k=k), hypothetical

docs_hyde_klue, hyp_klue = hyde_retrieve_klue(TEST_Q_KLUE)
print("가상 기사(HyDE):\n", hyp_klue[:300], "\n---")
print("첫 문서:", docs_hyde_klue[0].page_content[:200])

가상 기사(HyDE):
 최근 국내에서 해킹 사건이 발생하여 리플이 포함된 통장 수가 급증한 것으로 나타났다. 금융감독원에 따르면, 이번 해킹으로 인해 피해를 입은 통장 수는 약 1,200개로 추정되며, 이들 통장에서는 리플을 포함한 암호화폐가 무단으로 인출된 사례가 보고되었다. 관계 당국은 피해자들의 신고를 접수받아 조사에 착수했으며, 해킹 경로와 관련된 추가 정보를 확보하기 위해 사이버 수사에 나섰다. 금융업계는 이번 사건을 계기로 보안 시스템 강화와 함께 고객들의 주의가 필요하다고 강조하고 있다. 
---
첫 문서: 국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 20


### Step H. Multilingual Cross-encoder Reranker (KLUE)

메인 Step 4 의 `reranker` 인스턴스를 그대로 재사용합니다. (모델 재다운로드 불필요)

In [28]:
reranker_klue = reranker  # BAAI/bge-reranker-v2-m3 재사용

def rerank_klue(query, docs, top_k=3):
    """KLUE 뉴스 문단을 cross-encoder 로 재점수화해 상위 top_k 반환 (Step 4 rerank 와 동일 구조)"""
    if not docs:
        return []
    pairs = [(query, d.page_content) for d in docs]
    scores = reranker_klue.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [d for d, _ in ranked[:top_k]]

cand_klue = db_klue.as_retriever(search_kwargs={"k": 10}).invoke(TEST_Q_KLUE)
top3_klue = rerank_klue(TEST_Q_KLUE, cand_klue, top_k=3)
print(f"후보 {len(cand_klue)}개 → 상위 3개 선별")
print("최상위 문서:", top3_klue[0].page_content[:200])

후보 10개 → 상위 3개 선별
최상위 문서: 국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 20


### Step I. Advanced RAG 체인 (KLUE) — 넓게 → Rerank → LLM

In [29]:
def advanced_rag_klue(question):
    candidates = db_klue.as_retriever(search_kwargs={"k": 10}).invoke(question)
    top = rerank_klue(question, candidates, top_k=3)
    answer = (RAG_PROMPT_KLUE | llm | StrOutputParser()).invoke(
        {"context": format_docs(top), "question": question})
    return answer, top

ans_klue, ctx_klue = advanced_rag_klue(TEST_Q_KLUE)
print("Advanced RAG (KLUE) 답변:\n", ans_klue)
print("\n참고 문서 수:", len(ctx_klue))

Advanced RAG (KLUE) 답변:
 200여개의 계좌입니다.

참고 문서 수: 3


### Step J. RAGAS 로 Naive vs Advanced 비교 (KLUE)

In [30]:
naive_answers_k, naive_contexts_k = [], []
for q in questions_klue:
    ctx = naive_retriever_klue.invoke(q)
    a = (RAG_PROMPT_KLUE | llm | StrOutputParser()).invoke(
        {"context": format_docs(ctx), "question": q})
    naive_answers_k.append(a)
    naive_contexts_k.append([d.page_content for d in ctx])

adv_answers_k, adv_contexts_k = [], []
for q in questions_klue:
    a, ctx = advanced_rag_klue(q)
    adv_answers_k.append(a)
    adv_contexts_k.append([d.page_content for d in ctx])

naive_ds_klue = make_dataset(naive_answers_k, naive_contexts_k,
                             qs=questions_klue, refs=ground_truths_klue)
adv_ds_klue = make_dataset(adv_answers_k, adv_contexts_k,
                           qs=questions_klue, refs=ground_truths_klue)
print(f"KLUE 데이터셋 준비 완료 — {EVAL_N_KLUE}개 질문 × 2개 파이프라인")

KLUE 데이터셋 준비 완료 — 20개 질문 × 2개 파이프라인


In [31]:
naive_result_k = run_eval(naive_ds_klue, "KLUE Naive RAG")
adv_result_k   = run_eval(adv_ds_klue,   "KLUE Advanced RAG")

naive_df_k = naive_result_k.to_pandas()
adv_df_k   = adv_result_k.to_pandas()

compare_klue = pd.concat([summary(naive_df_k, "KLUE Naive"),
                          summary(adv_df_k,   "KLUE Advanced")], axis=1)
print(compare_klue.round(3))
print("\nDelta (Advanced - Naive):")
print((compare_klue["KLUE Advanced"] - compare_klue["KLUE Naive"]).round(3))

=== KLUE Naive RAG 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

=== KLUE Advanced RAG 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

                   KLUE Naive  KLUE Advanced
faithfulness            0.575           0.60
answer_relevancy        0.184           0.15
context_precision       0.667           0.75
context_recall          0.750           0.75

Delta (Advanced - Naive):
faithfulness         0.025
answer_relevancy    -0.034
context_precision    0.083
context_recall       0.000
dtype: float64


### KorQuAD(위키) vs KLUE-MRC(뉴스) 도메인 비교

In [32]:
domain_compare = pd.concat([
    summary(naive_df,   "KorQuAD Naive"),
    summary(adv_df,     "KorQuAD Advanced"),
    summary(naive_df_k, "KLUE Naive"),
    summary(adv_df_k,   "KLUE Advanced"),
], axis=1)
print(domain_compare.round(3))

                   KorQuAD Naive  KorQuAD Advanced  KLUE Naive  KLUE Advanced
faithfulness               0.600             0.850       0.575           0.60
answer_relevancy           0.261             0.284       0.184           0.15
context_precision          0.642             0.800       0.667           0.75
context_recall             0.700             0.800       0.750           0.75


### Step K. (선택) 통계적 신뢰도 — paired t-test

질문 20개로는 표본 분산이 커서 Naive vs Advanced 차이가 우연일 수 있습니다. 같은 질문에 두 파이프라인을 돌렸으므로 **대응표본 t-검정** 을 쓸 수 있습니다. (질문 수를 50~100 으로 늘리면 검정력이 올라가지만 토큰 비용도 함께 늘어납니다.)

In [33]:
def paired_test(df_a, df_b, label_a, label_b):
    print(f"{label_a} vs {label_b} — paired t-test")
    for m in METRIC_COLS:
        a = df_a[m].astype(float).fillna(0)
        b = df_b[m].astype(float).fillna(0)
        diff = b.mean() - a.mean()
        if (b - a).abs().sum() == 0:
            print(f"  {m:<18} Δ={diff:+.3f}  (동일 결과 — 검정 불가)")
            continue
        t, p = stats.ttest_rel(b, a)
        verdict = "유의미(p<0.05)" if p < 0.05 else "표본 noise 수준"
        print(f"  {m:<18} Δ={diff:+.3f}  t={t:+.2f}  p={p:.3f}  → {verdict}")

paired_test(naive_df,   adv_df,   "KorQuAD Naive", "KorQuAD Advanced")
print()
paired_test(naive_df_k, adv_df_k, "KLUE Naive",    "KLUE Advanced")

KorQuAD Naive vs KorQuAD Advanced — paired t-test
  faithfulness       Δ=+0.250  t=+2.52  p=0.021  → 유의미(p<0.05)
  answer_relevancy   Δ=+0.023  t=+0.65  p=0.525  → 표본 noise 수준
  context_precision  Δ=+0.158  t=+2.17  p=0.043  → 유의미(p<0.05)
  context_recall     Δ=+0.100  t=+1.45  p=0.163  → 표본 noise 수준

KLUE Naive vs KLUE Advanced — paired t-test
  faithfulness       Δ=+0.025  t=+0.33  p=0.748  → 표본 noise 수준
  answer_relevancy   Δ=-0.034  t=-1.87  p=0.077  → 표본 noise 수준
  context_precision  Δ=+0.083  t=+0.97  p=0.344  → 표본 noise 수준
  context_recall     Δ=+0.000  t=+0.00  p=1.000  → 표본 noise 수준


### 추가 실험 — `is_impossible` 케이스를 섞으면 어떤 지표가 망가지나

Step B 에서 걸러냈던 '답할 수 없는 질문' 5개를 평가 셋에 섞어, 어떤 지표가 무너지는지 직접 확인합니다. 정답 텍스트가 비어 있으므로 `reference` 는 빈 문자열로 둡니다.

In [34]:
impossible = ds_klue.filter(lambda x: x["is_impossible"]).shuffle(seed=42).select(range(5))
q_imp = [ex["question"] for ex in impossible]

ans_imp, ctx_imp = [], []
for q in q_imp:
    a, ctx = advanced_rag_klue(q)
    ans_imp.append(a)
    ctx_imp.append([d.page_content for d in ctx])

imp_ds = make_dataset(ans_imp, ctx_imp, qs=q_imp, refs=[""] * len(q_imp))
imp_result = run_eval(imp_ds, "KLUE is_impossible=True (5문항)")
imp_df = imp_result.to_pandas()
print(imp_df[METRIC_COLS].mean().round(3))
print("\n--- 답변 예시 ---")
for q, a in list(zip(q_imp, ans_imp))[:3]:
    print(f"Q: {q}\nA: {a}\n")

=== KLUE is_impossible=True (5문항) 채점 ===


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

faithfulness         0.0
answer_relevancy     0.0
context_precision    0.0
context_recall       NaN
dtype: float64

--- 답변 예시 ---
Q: 웹 드라마가 끝난 달은?
A: 기사에서 확인할 수 없습니다.

Q: 매해 연구개발 인력을 축소시키고 있다고 한 사람은?
A: 기사에서 확인할 수 없습니다.

Q: 가계 재무건전성이 매우 위험하다고 분석한 기관은?
A: 기사에서 확인할 수 없습니다.



### 마지막 Quiz — 정리

1. **도메인 비교**: KorQuAD(위키) 와 KLUE-MRC(뉴스) 중 어떤 지표가 가장 크게 달라졌나요? 뉴스 기사는 (1) 한 기사에 여러 사건·인용이 섞여 있고, (2) '지난 3일', '올해' 처럼 시점 표현이 상대적이며, (3) 숫자·기관명이 많아 **검색이 정답 문단을 찾아도 문단 안에서 정답 구절을 짚기 어려운** 특성이 있습니다. 그래서 위키보다 `context_precision`·`faithfulness` 의 변동이 커지는 경향이 있습니다.
2. **Advanced 효과**: 위 비교표의 Δ 를 KorQuAD 와 KLUE 사이에서 비교해 보세요. 문서가 길수록 Reranker 가 걸러줄 여지가 커집니다.
3. **`is_impossible` 케이스**: 위 추가 실험에서 보듯 `context_recall` 과 `context_precision` 이 가장 먼저 무너집니다. 정답(reference)이 없으니 '정답을 담은 컨텍스트' 자체를 정의할 수 없기 때문입니다. 실제 서비스라면 '모른다' 로 답하는 능력을 따로 (abstention accuracy 등으로) 평가해야 합니다.
4. **MIRACL ko**: 위키 문단 검색 코퍼스가 훨씬 커서(수십만 문단) `context_recall` 이 더 이상 포화되지 않고, Multi-Query·RAG-Fusion 같은 recall 확장 기법의 효과가 Reranker 만큼 커질 것으로 예상됩니다.

## 마치며

- **KorQuAD v1** 위에 Naive RAG 베이스라인 구성
- Multi-Query / **RAG-Fusion (RRF 직접 구현)** / HyDE / Cross-encoder Reranking 적용
- '넓게 검색 → Reranker 로 좁힘 → LLM 답변' Advanced RAG 체인 조립
- **Self-RAG** — 검색 필요성 판단 + 답변 자가 비평 + HyDE 재시도
- RAGAS 4대 지표로 Naive / RAG-Fusion / Advanced 정량 비교 + paired t-test
- 도메인을 옮긴 **KLUE-MRC (뉴스)** 에서 같은 파이프라인 재구성 및 `is_impossible` 케이스 추가 실험

## 실행 결과 요약 · 회고

이 노트북을 **처음부터 끝까지 한 번에 실행**한 결과입니다. (Judge LLM: `gpt-4o-mini`, 질문 20개 × 파이프라인, 검색 코퍼스: KorQuAD unique context 847개 / KLUE-MRC unique context 199개)

### KorQuAD v1 (위키)

| 지표 | Naive RAG | RAG-Fusion | Advanced RAG | Δ(Adv-Naive) |
|---|---|---|---|---|
| faithfulness | 0.600 | 0.725 | **0.850** | **+0.250** (p=0.021) |
| answer_relevancy | 0.261 | 0.269 | **0.284** | +0.023 (p=0.525) |
| context_precision | 0.642 | 0.692 | **0.800** | **+0.158** (p=0.043) |
| context_recall | 0.700 | **0.800** | **0.800** | +0.100 (p=0.163) |

### KLUE-MRC (뉴스)

| 지표 | Naive RAG | Advanced RAG | Δ |
|---|---|---|---|
| faithfulness | 0.575 | 0.600 | +0.025 |
| answer_relevancy | 0.184 | 0.150 | -0.034 |
| context_precision | 0.667 | 0.750 | +0.083 |
| context_recall | 0.750 | 0.750 | +0.000 |

### 읽는 법 / 배운 점

1. **Reranker 효과가 `context_precision` 에서 그대로 확인됩니다.** KorQuAD 에서 +0.158 (paired t-test p=0.043) 로 표본 20개에서도 유의미했습니다. Step 4 의 순위 변화 출력에서도 임베딩 3·5·10위였던 문단이 rerank 후 2·3·4위로 올라오는 것을 볼 수 있습니다.
2. **`faithfulness` 는 오히려 크게 올랐습니다(+0.250).** 강의 노트에서는 컨텍스트가 좁아지며 살짝 내려갈 수 있다고 했지만, 이번 코퍼스(847개 unique context)에서는 관련 없는 문단이 빠지면서 LLM 이 근거 없는 문장을 덜 만들었습니다.
3. **`answer_relevancy` 는 0.15~0.28 로 낮습니다.** KorQuAD/KLUE 정답이 '대중교통체계' 처럼 한 구절이라 RAGAS 가 답변에서 질문을 역추론할 때 흐려지는, **데이터셋의 구조적 특성**입니다. 절대값이 아니라 Naive 대비 상대 변화로 읽어야 합니다.
4. **RAG-Fusion(RRF)은 recall 쪽에서 이득이 납니다.** context_recall 0.700 → 0.800 으로 Advanced 와 같은 수준까지 올라갔지만, precision 개선폭(+0.050)은 Reranker(+0.158)보다 작았습니다. 즉 **RRF 는 '넓히기', Reranker 는 '좁히기'** 로 역할이 갈립니다.
5. **뉴스 도메인(KLUE)에서는 개선폭이 전부 표본 noise 수준(p>0.05)이었습니다.** 뉴스 기사는 한 문서에 여러 사건·인용·숫자가 섞여 있어 '정답 문단을 찾아도 정답 구절을 짚기 어렵다' 는 점이 위키와 다릅니다. 그만큼 Reranker 로 문단을 골라도 남는 이득이 작았습니다.
6. **`is_impossible=True` 5문항 추가 실험**: 파이프라인은 모두 "기사에서 확인할 수 없습니다" 로 올바르게 답했지만, RAGAS 점수는 faithfulness/answer_relevancy/context_precision = 0.0, context_recall = NaN 으로 전부 무너졌습니다. **reference 가 비어 있으면 RAGAS 4대 지표는 정의되지 않습니다.** 답변 거부 능력은 abstention accuracy 같은 별도 지표로 평가해야 합니다.

### 시행착오 기록

- **Self-RAG 자가 비평 프롬프트**: 처음에는 *"모든 주장이 문서로 뒷받침되는지 엄격히 판단하라"* 로 썼더니, `대중교통체계입니다.` 처럼 **정답인 짧은 답변까지 NOT_SUPPORTED** 로 판정해 HyDE 재검색이 매번 발동했습니다. *"한 단어·한 구절처럼 짧아도 문서에서 확인되면 SUPPORTED, 질문에 대한 완결성이 아니라 문서 근거 여부만 본다"* 로 기준을 명시한 뒤 정상 동작했습니다. (근거 없는 답변에는 여전히 NOT_SUPPORTED 를 반환하는지 별도로 확인)
- **KLUE 임베딩 적재**: 뉴스 context 는 평균 토큰 수가 커서 한 번에 넘기면 OpenAI embeddings 의 300k 토큰/요청 한도에 걸립니다. 100개씩 batch 로 `add_documents` 했습니다.
- **컬렉션 분리**: `db` 와 `db_klue` 를 `collection_name` 으로 분리해 두 도메인 결과가 섞이지 않게 했습니다.
- **키 관리**: API 키는 노트북에 하드코딩하지 않고 Colab `userdata` / 환경 변수 `OPENAI_API_KEY` 에서 읽도록 했습니다.

### 다음에 해볼 것

- 질문 수를 50~100 으로 늘려 KLUE 에서도 유의미한 차이가 나오는지 확인
- `Alibaba-NLP/gte-multilingual-reranker-base` 등 다른 다국어 reranker 와 비교
- MIRACL ko 처럼 코퍼스가 큰 데이터셋에서 Multi-Query/RRF 의 recall 효과 재측정
